In [1]:
import numpy as np
import rasterio as rio
from scipy.stats import mode
from matplotlib import pyplot as plt
from tqdm import tqdm
import itertools
from affine import Affine
from rasterio.enums import Resampling
from rasterio.warp import reproject

In [ ]:
global_lv1_ = rio.open(r'../data/a-global-map-of-terrestrial-habitat-types/iucn_habitatclassification_composite_lvl1_ver004.tif')
# Source: Jung et al. (2020) A global map of terrestrial habitat types, Zenodo. (https://zenodo.org/records/4058819)

In [4]:
global_lv1_.shape

(200376, 400752)

In [3]:
def block_mode_fast(arr, block_size):
    H, W = arr.shape
    r = block_size
    assert H % r == 0 and W % r == 0, "Dimensions must be divisible by block size"

    # reshape to (H//r, r, W//r, r)
    reshaped = arr.reshape(H // r, r, W // r, r)
    # move axes to (H//r, W//r, r, r)
    blocks = reshaped.transpose(0, 2, 1, 3)
    # flatten to (H//r, W//r, r*r)
    flat_blocks = blocks.reshape(H // r, W // r, -1)

    max_val = arr.max()
    modes = np.empty((H // r, W // r), dtype=arr.dtype)
    for i in range(H // r):
        for j in range(W // r):
            modes[i, j] = np.bincount(flat_blocks[i, j], minlength=max_val + 1).argmax()
    return modes

In [4]:
H, W = global_lv1_.shape
k = 10000
r = 10
q = int(k // r)
I = H // k + 1
J = W // k + 1 

new_array = np.zeros((I * q, J * q), dtype=np.uint16)

for idx in tqdm(itertools.product(range(I), range(J))):
    i, j = idx
    m = k if i < (I-1) else H - i * k
    n = k if j < (J-1) else W - j * k
    a_window = ((i*k, i*k+m), (j*k, j*k+n))
    a_slice = global_lv1_.read(1, window=a_window)

    aa = np.vstack([a_slice, np.zeros((k - a_slice.shape[0], a_slice.shape[1]))]).astype(np.uint16)
    bb = np.hstack([aa, np.zeros((aa.shape[0], k - a_slice.shape[1]))]).astype(np.uint16)
    cc = np.where(bb > 0, bb, 0).astype(np.uint16)
    mode_result = block_mode_fast(cc, block_size=10)

    new_array[i*q:(i+1)*q, j*q:(j+1)*q] = mode_result 
    # print(a_window)

861it [2:13:03,  9.27s/it]


In [5]:
the_transform = Affine(global_lv1_.transform[0]*10, 0.0, global_lv1_.transform[2], 0.0, -global_lv1_.transform[0]*10, global_lv1_.transform[5])

In [ ]:
with rio.open(
    '../results/terrestrial-habitat-class-maps/habitat-classifications-lv1.tif', 
    'w', 
    driver='GTiff', 
    height=20038, 
    width=40076, 
    count=1, 
    dtype=new_array.dtype, 
    crs='EPSG:4326', 
    transform=the_transform, 
    compress='lzw'
    ) as dst:
    dst.write(new_array[:20038, :40076], 1)

reprojection to EckertIV of 1km resolution

In [ ]:
habitat_classifications_lv1_= rio.open(r'../results/terrestrial-habitat-class-maps/habitat-classifications-lv1.tif')

In [9]:
habitat_classifications_lv1 = habitat_classifications_lv1_.read(1)

In [ ]:
country_raster_1km_ = rio.open(r"../results/country-rasters/country-raster-eck4-1km.tif")

In [11]:
habitat_classifications_lv1_1km = np.zeros(country_raster_1km_.shape, dtype=np.uint16)
reproject(
        source=habitat_classifications_lv1,
        destination=habitat_classifications_lv1_1km,
        src_transform=habitat_classifications_lv1_.transform,
        src_crs='EPSG:4326',
        dst_transform=country_raster_1km_.transform,
        dst_crs=country_raster_1km_.crs,
        resampling=Resampling.mode
    )

(array([[0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        ...,
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0]], dtype=uint16),
 Affine(1000.0, 0.0, -16922000.0,
        0.0, -1000.0, 8461000.0))

In [ ]:
with rio.open(
    '../results/terrestrial-habitat-class-maps/habitat-classifications-lv1-1km.tif', 
    'w', 
    driver='GTiff', 
    height=habitat_classifications_lv1_1km.shape[0], 
    width=habitat_classifications_lv1_1km.shape[1], 
    count=1, 
    dtype=habitat_classifications_lv1_1km.dtype, 
    crs=country_raster_1km_.crs, 
    transform=country_raster_1km_.transform, 
    compress='lzw'
    ) as dst:
    dst.write(habitat_classifications_lv1_1km, 1)